In [1]:
import torch
from gendyndiff.common.data.collate import CustomCollate
from gendyndiff.diffusion.data.batched_data import CustomCrystalDataset
from gendyndiff.diffusion.diffusion_module import DiffusionModule
from gendyndiff.diffusion.losses import DenoisingScoreMatchingLoss
from gendyndiff.diffusion.timestep_samplers import UniformTimestepSampler
from gendyndiff.diffusion.corruption.sde_lib import VPSDE
from gendyndiff.diffusion.corruption.multi_corruption import MultiCorruption
from gendyndiff.common.gemnet.gemnet import GemNetT

MODELS_PROJECT_ROOT: /home/agore/GenDynDiff/gendyndiff


In [2]:
dump_file_path = "/home/agore/GenDynDiff/datasets/SrTiO3/dump.NPT"
cfg_file_path = "/home/agore/GenDynDiff/datasets/SrTiO3/SrTiO3_supercell_555.cfg"

custom_collate = CustomCollate()
datasets = CustomCrystalDataset.from_dump_file(dump_file_path, cfg_file_path)

In [3]:
model_targets = {
    "velocities": "score_times_std",  # Only target velocities
    "positions": "zero",
    "lattice": "zero",
    "atomic_types": "zero"
}

weights = {
    "positions": 1.0,    # Position loss only
    "velocities": 0.0,
    "lattice": 0.0,
    "atomic_types": 0.0
}
loss_fn = DenoisingScoreMatchingLoss(
    model_targets=model_targets,
    weights=weights,
)

In [4]:
velocity_corruption = VPSDE(beta_min=0.1, beta_max=20)
corruption = MultiCorruption(
    sdes={"velocities": velocity_corruption},
)

In [5]:
class AtomEmbedding(torch.nn.Module):
    def __init__(self, emb_size, max_atomic_number=128):
        super().__init__()
        self.emb_size = emb_size
        self.embedding = torch.nn.Embedding(max_atomic_number + 1, emb_size)
    def forward(self, atomic_numbers):
        return self.embedding(atomic_numbers)

atom_embedding = AtomEmbedding(emb_size=128, max_atomic_number=128)
score_model = GemNetT(
    atom_embedding=atom_embedding,
    num_targets=3,
    latent_dim=128,
    num_spherical=7,
    num_radial=128,
    num_blocks=3,
    emb_size_atom=512,
    emb_size_edge=512,
    emb_size_trip=64,
    emb_size_rbf=16,
    emb_size_cbf=16,
    cutoff=6.0,
    activation="swish",
    max_neighbors=50,
)

In [6]:
diffusion_module = DiffusionModule(
    model=score_model,
    corruption=corruption,
    loss_fn=loss_fn,
    timestep_sampler=UniformTimestepSampler(min_t=1e-5, max_t=1.0),
)

ValueError: 'zero' is not a valid ModelTarget

In [6]:
prev_batch = None
for timestep_idx, dataset in enumerate(datasets):
    print(f"Timestep {timestep_idx + 1}:")
    current_batch = custom_collate([dataset])
    custom_collate.print_atoms(current_batch)
    print("-" * 50)

    if prev_batch is not None:
        # Corrupt velocities in previous batch
        noisy_batch, t = diffusion_module._corrupt_batch(prev_batch)

        # Forward pass to denoise velocities
        reconstructed = diffusion_module.model(noisy_batch, t)

        delta_t = 0.5  # Match your MD timestep
        predicted_positions = prev_batch.positions + reconstructed.velocities * delta_t

        # Calculate position loss against current batch
        pos_diff = predicted_positions - current_batch.positions
        cartesian_diff = torch.matmul(pos_diff, current_batch.lattice)
        loss = torch.mean(cartesian_diff**2)

        print(f"Position MSE Loss: {loss.item():.4e}")
        print("-" * 50)

    prev_batch = current_batch

NameError: name 'datasets' is not defined